# Phase 1 · Exercise 1 — What are the labels?

**Why this notebook:** you cannot model a target you do not understand. Before touching any EEG signal,
we look only at `train.csv`: who labelled what, how many experts voted, how much they agreed,
and how good a model is that *ignores the EEG completely*. That last number is the floor every later model must beat.

**What you will have at the end**

| Output | Used in the writeup for |
|---|---|
| Dataset counts (rows, EEGs, patients, windows per patient) | Background / Methods |
| Histogram of the number of votes per window | Background (two labelling rounds) |
| Class distribution, < 10 vs ≥ 10 votes | Background, motivates Phase 2 Step 3 |
| Idealized / proto / edge split | Background (Jing et al. 2023) |
| KL of the uniform and class-prior baselines | Results — first rows of the score table |

All numbers are written to `outputs_01/` so you can copy them into the writeup and the score log.

**Runs on:** Kaggle (attach the competition data) or locally with the data in `data/hms/` at the repo root. Only `train.csv` is read, so it takes seconds.

## 0. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

# Find the competition data. On Kaggle the mount folder name varies, so search all of /kaggle/input
# for a train.csv that sits next to a train_spectrograms folder; locally use data/hms (repo README).
def find_data_dir():
    kaggle = Path("/kaggle/input")
    if kaggle.exists():
        for csv in sorted(kaggle.rglob("train.csv")):
            if (csv.parent / "train_spectrograms").exists():
                return csv.parent
    for p in [Path("../data/hms"), Path("data/hms")]:
        if (p / "train.csv").exists():
            return p
    return None


DATA_DIR = find_data_dir()
if DATA_DIR is None:
    attached = sorted(str(p) for p in Path("/kaggle/input").glob("*/*")) if Path("/kaggle/input").exists() else []
    raise FileNotFoundError(
        "Competition data not found. On Kaggle: right panel -> 'Add Input' -> Competitions -> "
        "'HMS - Harmful Brain Activity Classification' -> Add. Locally: put the data in data/hms/.\n"
        f"Currently under /kaggle/input: {attached or 'nothing'}")
print("Data directory:", DATA_DIR.resolve())

OUT_DIR = Path("/kaggle/working/outputs_01") if Path("/kaggle/working").exists() else Path("outputs_01")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Outputs go to:  ", OUT_DIR.resolve())

# The six vote columns, in the order required by the submission file.
VOTE_COLS = ["seizure_vote", "lpd_vote", "gpd_vote", "lrda_vote", "grda_vote", "other_vote"]
CLASS_NAMES = ["Seizure", "LPD", "GPD", "LRDA", "GRDA", "Other"]
HIGH_VOTES = 10          # threshold used throughout the project for "reliable" labels
SEED = 42

pd.set_option("display.width", 140)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 1. Load `train.csv` and count

One row of `train.csv` = one **labelled 10-second window** (`label_id`), taken from the centre of a 50-s EEG
segment. One EEG recording (`eeg_id`) can have many windows (`eeg_sub_id`, with different `eeg_label_offset_seconds`),
and one patient can have many recordings.

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
print(train.shape)
print(train.dtypes.to_string())
train.head()

In [ ]:
# Sanity checks: no missing values, label_id is unique, vote columns are non-negative integers.
print("Missing values per column:", train.isna().sum().sum())
print("label_id unique:          ", train.label_id.is_unique)
print("Votes all >= 0 integers:  ", bool((train[VOTE_COLS] >= 0).all().all()
                                         and np.allclose(train[VOTE_COLS], train[VOTE_COLS].round())))

In [ ]:
counts = pd.Series({
    "windows (rows)": len(train),
    "unique label_id": train.label_id.nunique(),
    "unique eeg_id": train.eeg_id.nunique(),
    "unique spectrogram_id": train.spectrogram_id.nunique(),
    "unique patient_id": train.patient_id.nunique(),
})
print(counts.to_string())

win_per_eeg = train.groupby("eeg_id").size()
win_per_patient = train.groupby("patient_id").size()
eeg_per_patient = train.groupby("patient_id").eeg_id.nunique()

summary = pd.DataFrame({
    "windows per eeg_id": win_per_eeg.describe(),
    "windows per patient": win_per_patient.describe(),
    "eeg_ids per patient": eeg_per_patient.describe(),
}).round(1)
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(win_per_eeg, bins=np.logspace(0, np.log10(win_per_eeg.max() + 1), 40), color="tab:blue")
axes[0].set(xscale="log", yscale="log", xlabel="windows per eeg_id", ylabel="number of EEGs",
            title="Windows per EEG recording")
axes[1].hist(win_per_patient, bins=np.logspace(0, np.log10(win_per_patient.max() + 1), 40), color="tab:orange")
axes[1].set(xscale="log", yscale="log", xlabel="windows per patient", ylabel="number of patients",
            title="Windows per patient")
plt.tight_layout()
plt.savefig(OUT_DIR / "windows_per_eeg_and_patient.png", bbox_inches="tight")
plt.show()

# How concentrated is the data? Share of all windows that come from the 1% / 10% largest patients.
sorted_w = win_per_patient.sort_values(ascending=False)
for frac in [0.01, 0.10]:
    k = max(1, int(round(frac * len(sorted_w))))
    print(f"Top {frac:.0%} of patients ({k}) contribute {sorted_w.iloc[:k].sum() / len(train):.1%} of all windows")

**What to note (write in your own words):**
- A few patients contribute a large share of the windows. If the same patient is in train *and* validation,
  the model can recognise the patient instead of the pattern → the score looks better than it is.
  This is why we **always** split by `patient_id` (GroupKFold).
- Many windows per `eeg_id`: consecutive windows of one recording often overlap and share the same label
  (checked in section 5). This feeds **Decision Point A** (one window per EEG vs all windows).

## 2. How many experts voted? (total votes per window)

The data were labelled in two rounds: most windows got only a few votes, while a subset was labelled by many experts.
We expect a **bimodal** histogram with (almost) nothing in between.

In [ ]:
train["total_votes"] = train[VOTE_COLS].sum(axis=1)
vote_counts = train.total_votes.value_counts().sort_index()
print(vote_counts.to_string())

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(vote_counts.index, vote_counts.values, color="tab:blue")
ax.axvline(HIGH_VOTES - 0.5, color="k", ls="--", lw=1)
ax.text(HIGH_VOTES - 0.3, ax.get_ylim()[1] * 0.9, f"≥ {HIGH_VOTES} votes", va="top")
ax.set(xlabel="total number of expert votes", ylabel="number of windows", title="Votes per labelled window")
plt.tight_layout()
plt.savefig(OUT_DIR / "votes_histogram.png", bbox_inches="tight")
plt.show()

hi = train.total_votes >= HIGH_VOTES
print(f"Windows with 8-9 votes: {train.total_votes.between(8, 9).sum()}")
print(f"Windows with >= {HIGH_VOTES} votes: {hi.sum()} ({hi.mean():.1%} of windows), "
      f"{train.loc[hi, 'eeg_id'].nunique()} EEGs, {train.loc[hi, 'patient_id'].nunique()} patients")
print(f"Windows with <  {HIGH_VOTES} votes: {(~hi).sum()} ({(~hi).mean():.1%} of windows), "
      f"{train.loc[~hi, 'eeg_id'].nunique()} EEGs, {train.loc[~hi, 'patient_id'].nunique()} patients")

**What to note:** where exactly the gap is, and what fraction of windows/patients sit in each group.
A window with 1–3 votes has a "soft" label that is really a hard (one-hot) label decided by one or two people;
a window with 15 votes has a real distribution. The two groups are different kinds of target.

## 3. Soft labels and class distribution

The target is the vote distribution: $y_c = \text{votes}_c / \sum_k \text{votes}_k$, a vector of 6 numbers that sums to 1.
`expert_consensus` in the file is the class with the most votes.

In [ ]:
Y = train[VOTE_COLS].to_numpy(float) / train.total_votes.to_numpy(float)[:, None]
assert np.allclose(Y.sum(axis=1), 1), "every soft label must sum to 1"
soft = pd.DataFrame(Y, columns=CLASS_NAMES, index=train.index)

# Does the argmax of our soft label agree with the provided expert_consensus column?
argmax_name = soft.idxmax(axis=1)
ties = (soft.values == soft.values.max(axis=1, keepdims=True)).sum(axis=1) > 1
print("expert_consensus values:", sorted(train.expert_consensus.unique()))
print(f"argmax == expert_consensus on non-tied rows: {(argmax_name == train.expert_consensus)[~ties].mean():.4f}")
print(f"rows with a tie for the top class: {ties.sum()}")
soft.head()

In [ ]:
groups = {f"< {HIGH_VOTES} votes": ~hi, f">= {HIGH_VOTES} votes": hi, "all windows": hi | ~hi}

# Two views of "class distribution":
#  (a) consensus share = fraction of windows whose majority class is c (hard view)
#  (b) mean soft label = average vote fraction for c (soft view; this is also the class prior used in section 6)
consensus_share = pd.DataFrame({g: train.loc[m, "expert_consensus"].value_counts(normalize=True)
                                for g, m in groups.items()}).reindex(CLASS_NAMES)
mean_soft = pd.DataFrame({g: soft[m].mean() for g, m in groups.items()})

print("(a) Share of windows by expert_consensus")
print(consensus_share.round(3).to_string())
print("\n(b) Mean soft label (class prior)")
print(mean_soft.round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
x = np.arange(len(CLASS_NAMES))
for ax, table, title in [(axes[0], consensus_share, "(a) share by majority class"),
                         (axes[1], mean_soft, "(b) mean soft label")]:
    for i, g in enumerate(list(groups)[:2]):
        ax.bar(x + (i - 0.5) * 0.4, table[g], width=0.4, label=g)
    ax.set(xticks=x, xticklabels=CLASS_NAMES, title=title)
axes[0].set_ylabel("fraction")
axes[0].legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "class_distribution_low_vs_high.png", bbox_inches="tight")
plt.show()

**What to note:** which classes are much more common in the low-vote group than in the high-vote group (and vice versa).
If the two distributions differ, a model trained on all windows learns a prior that does not match the
high-vote windows — one reason why top solutions trained on all data and then fine-tuned on high-vote data (Phase 2, Step 3).

## 4. How much do experts agree? Idealized / proto / edge

Jing et al. (2023, *Neurology*) describe a continuum between patterns:
- **idealized**: (almost) all experts pick the same class,
- **proto-pattern**: a clear majority for one class, with a minority for a neighbouring class,
- **edge case**: experts split roughly evenly between two classes.

The paper does not give numeric thresholds for this dataset, so the rule below is **my own operationalisation**
(state it as such in the writeup). With $p_1 \ge p_2$ the two largest vote fractions:

| category | rule |
|---|---|
| idealized | $p_1 \ge 0.9$ |
| edge | $p_1 - p_2 \le 0.2$ |
| proto | everything else |

This is only meaningful when there are enough votes: with 1 vote every window is trivially "idealized",
so we categorise the **≥ 10-vote** windows only.

In [ ]:
IDEAL_MIN, EDGE_MAX_GAP = 0.9, 0.2

sorted_p = np.sort(Y, axis=1)[:, ::-1]
p1, p2 = sorted_p[:, 0], sorted_p[:, 1]
train["p1"], train["p2"] = p1, p2
train["agreement"] = np.select([p1 >= IDEAL_MIN, p1 - p2 <= EDGE_MAX_GAP], ["idealized", "edge"], "proto")
# Name of the second-most-voted class (useful to see which classes get confused).
train["second_class"] = np.array(CLASS_NAMES)[np.argsort(-Y, axis=1, kind="stable")[:, 1]]

H = train[hi]
agree_tab = pd.crosstab(H.expert_consensus, H.agreement, margins=True).reindex(CLASS_NAMES + ["All"])
agree_tab = agree_tab[["idealized", "proto", "edge", "All"]]
print(f"Agreement category counts, >= {HIGH_VOTES}-vote windows")
agree_tab

In [ ]:
# Row-normalised version: within each majority class, what fraction is idealized / proto / edge?
(agree_tab.drop(columns="All").div(agree_tab["All"], axis=0)).round(2)

In [ ]:
# For edge cases: between which two classes are the experts split?
edge = H[H.agreement == "edge"]
pairs = edge.apply(lambda r: " / ".join(sorted([r.expert_consensus, r.second_class])), axis=1)
pairs.value_counts().head(10)

In [ ]:
# Mean agreement p1 per majority class (higher = experts agree more on this class).
H.groupby("expert_consensus").p1.agg(["mean", "median", "count"]).reindex(CLASS_NAMES).round(2)

### Three examples of each category

We have not looked at signals yet (that is Exercise 2), so the examples are shown as their vote distributions.
Write down their `eeg_id` / `label_id`: in Exercise 2 you can plot the EEG of exactly these windows.
To get a fair picture, the examples are taken from three different majority classes where possible.

In [ ]:
def pick_examples(df, n=3, seed=SEED):
    # One random window per majority class, from n different classes (falls back to any rows).
    by_class = df.groupby("expert_consensus").sample(1, random_state=seed)
    pick = by_class.sample(min(n, len(by_class)), random_state=seed)
    if len(pick) < n:
        pick = pd.concat([pick, df.drop(pick.index).sample(n - len(pick), random_state=seed)])
    return pick


examples = pd.concat({cat: pick_examples(H[H.agreement == cat]) for cat in ["idealized", "proto", "edge"]})
examples = examples.reset_index(level=0).rename(columns={"level_0": "category"})

fig, axes = plt.subplots(3, 3, figsize=(12, 7), sharey=True)
for ax, (_, row) in zip(axes.ravel(), examples.iterrows()):
    votes = row[VOTE_COLS].to_numpy(float)
    ax.bar(CLASS_NAMES, votes / votes.sum(), color="tab:gray")
    ax.bar(row.expert_consensus, row.p1, color="tab:red")
    ax.set_title(f"{row.category}: {row.expert_consensus}\neeg_id {row.eeg_id}, {int(row.total_votes)} votes",
                 fontsize=9)
    ax.tick_params(axis="x", labelsize=8)
for ax in axes[:, 0]:
    ax.set_ylabel("vote fraction")
plt.tight_layout()
plt.savefig(OUT_DIR / "agreement_examples.png", bbox_inches="tight")
plt.show()

examples_out = examples[["category", "label_id", "eeg_id", "eeg_label_offset_seconds", "spectrogram_id",
                         "spectrogram_label_offset_seconds", "patient_id", "expert_consensus",
                         "total_votes"] + VOTE_COLS]
examples_out.to_csv(OUT_DIR / "agreement_examples.csv", index=False)
examples_out

**What to note:** which classes are mostly idealized and which are mostly proto/edge, and which class pairs form
the edge cases. Compare with the clinical key: L vs G (one side vs both sides) and PD vs RDA (blips with gaps vs
continuous rhythm). Pairs like LPD/GPD or LRDA/GRDA mean experts disagree on *laterality*; LPD/LRDA or GPD/GRDA
mean they disagree on *periodic vs rhythmic*. Expert disagreement is part of the target — a perfect model
should predict it, not hide it.

## 5. Do windows of the same EEG share one label?

Input for **Decision Point A**. If most EEGs have the same vote distribution in all their windows,
the many windows per EEG are mostly near-duplicates (overlapping 50-s segments with one shared annotation).

In [ ]:
n_distinct = (train.groupby("eeg_id")[VOTE_COLS]
              .apply(lambda g: len(g.drop_duplicates())))
multi = win_per_eeg > 1
print(f"EEGs with more than one window: {multi.sum()} of {len(win_per_eeg)}")
print(f"... of those, all windows have the same votes: {(n_distinct[multi] == 1).mean():.1%}")
print(f"Distinct vote vectors per EEG (EEGs with >1 window):")
print(n_distinct[multi].value_counts().sort_index().head(10).to_string())

**What to note:** if this percentage is high, using all windows mostly adds repeated labels (plus slightly shifted
signal), while one window per EEG gives a smaller but less redundant set. Either way, GroupKFold by patient
keeps the evaluation honest.

## 6. First numbers: KL divergence of label-only baselines

**Metric.** For one window with true distribution $y$ and prediction $\hat y$:
$$\mathrm{KL}(y\,\|\,\hat y) = \sum_{c} y_c \log \frac{y_c}{\hat y_c},$$
with $0 \log 0 = 0$, averaged over windows. Lower is better, 0 is perfect.
Predictions are clipped to a small $\varepsilon$ and renormalised so that $\log 0$ never happens.
(The course template uses $\varepsilon = 10^{-15}$; clipping at $10^{-6}$ is a safer choice for real submissions.
For the two baselines below nothing is near 0, so $\varepsilon$ does not matter.)

**Baselines (no EEG used at all):**
- **(a) uniform**: predict 1/6 for every class.
- **(b) class prior**: predict the mean soft label of the *training* windows. The mean soft label is the
  constant prediction with the lowest average KL (minimising $-\sum_i\sum_c y_{ic}\log q_c$ over $q$ gives $q = \bar y$),
  so this is the best a model can do without looking at the signal.

To be honest the prior must be estimated on training data only. We do this with **5-fold GroupKFold on `patient_id`**,
the same split we will use for every model later.

In [ ]:
def kl_divergence(y_true, y_pred, eps=1e-15):
    # Mean over rows of KL(y_true || y_pred). Rows of y_pred are clipped to >= eps and renormalised.
    y_true = np.asarray(y_true, float)
    p = np.clip(np.asarray(y_pred, float), eps, 1)
    p = p / p.sum(axis=1, keepdims=True)
    safe_t = np.where(y_true > 0, y_true, 1.0)          # avoids log(0); those terms are multiplied by 0
    return float(np.mean(np.sum(np.where(y_true > 0, y_true * np.log(safe_t / p), 0.0), axis=1)))


# Sanity checks of the metric itself
uniform6 = np.full((1, 6), 1 / 6)
assert abs(kl_divergence(uniform6, uniform6)) < 1e-12                                 # identical -> 0
assert abs(kl_divergence(np.eye(6)[[0]], uniform6) - np.log(6)) < 1e-12              # one-hot vs uniform -> log 6
assert kl_divergence(Y[:100], Y[:100]) < 1e-12
print("metric checks passed; log(6) =", round(np.log(6), 4))

In [ ]:
def cv_prior_oof(Y, groups, train_mask=None, n_splits=5):
    # Out-of-fold class-prior predictions: for each fold, the mean soft label of the training folds.
    # train_mask (optional) restricts which training-fold windows are used to estimate the prior.
    if train_mask is None:
        train_mask = np.ones(len(Y), bool)
    oof = np.zeros_like(Y)
    for tr, va in GroupKFold(n_splits=n_splits).split(Y, groups=groups):
        assert not set(groups[tr]) & set(groups[va]), "a patient appears in two folds"
        tr = tr[train_mask[tr]]
        oof[va] = Y[tr].mean(axis=0)
    return oof


patients = train.patient_id.to_numpy()
hi_mask = hi.to_numpy()

preds = {
    "uniform": np.full_like(Y, 1 / 6),
    "prior (in-sample, all windows)": np.tile(Y.mean(axis=0), (len(Y), 1)),
    "prior (CV, fit on all windows)": cv_prior_oof(Y, patients),
    f"prior (CV, fit on >= {HIGH_VOTES}-vote windows)": cv_prior_oof(Y, patients, train_mask=hi_mask),
}

rows = []
for name, P in preds.items():
    rows.append({"baseline": name,
                 "KL all windows": kl_divergence(Y, P),
                 f"KL < {HIGH_VOTES} votes": kl_divergence(Y[~hi_mask], P[~hi_mask]),
                 f"KL >= {HIGH_VOTES} votes": kl_divergence(Y[hi_mask], P[hi_mask])})
baselines = pd.DataFrame(rows).set_index("baseline").round(4)
baselines

### Same baselines at the EEG level (to compare with the course template)

The course template sums the votes of all windows of an `eeg_id` into one row per EEG before computing its
`prior=` CV score. Recompute the CV prior that way, so that in Phase 2 Step 1 you can check that your KL function
gives the same number as the template's log.

In [ ]:
eeg = train.groupby("eeg_id").agg(patient_id=("patient_id", "first"), **{c: (c, "sum") for c in VOTE_COLS})
eeg_votes = eeg[VOTE_COLS].to_numpy(float)
Y_eeg = eeg_votes / eeg_votes.sum(axis=1, keepdims=True)
hi_eeg = eeg_votes.sum(axis=1) >= HIGH_VOTES

oof_eeg = cv_prior_oof(Y_eeg, eeg.patient_id.to_numpy())
eeg_rows = pd.DataFrame([
    {"baseline": "uniform (EEG level)", "KL all": kl_divergence(Y_eeg, np.full_like(Y_eeg, 1 / 6)),
     f"KL >= {HIGH_VOTES} votes": kl_divergence(Y_eeg[hi_eeg], np.full_like(Y_eeg[hi_eeg], 1 / 6))},
    {"baseline": "prior CV (EEG level)", "KL all": kl_divergence(Y_eeg, oof_eeg),
     f"KL >= {HIGH_VOTES} votes": kl_divergence(Y_eeg[hi_eeg], oof_eeg[hi_eeg])},
]).set_index("baseline").round(4)
print(f"{len(eeg)} EEGs, {hi_eeg.sum()} with >= {HIGH_VOTES} summed votes")
print("Note: the template's fold assignment may differ, so expect a close but not necessarily identical number.")
eeg_rows

In [ ]:
# Why are the low-vote windows so much worse? Per-window KL of the CV prior, by number of votes.
P = preds["prior (CV, fit on all windows)"]
per_window = np.sum(np.where(Y > 0, Y * np.log(np.where(Y > 0, Y, 1) / P), 0), axis=1)
by_votes = pd.Series(per_window).groupby(train.total_votes.clip(upper=20)).mean()

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.bar(by_votes.index, by_votes.values, color="tab:purple")
ax.set(xlabel="total votes (20 = 20 or more)", ylabel="mean KL of class prior",
       title="Prior baseline error vs number of votes")
plt.tight_layout()
plt.savefig(OUT_DIR / "prior_kl_by_votes.png", bbox_inches="tight")
plt.show()

**What to note:** a window with one vote has a one-hot target, so *any* spread-out prediction pays a large KL;
windows with many votes have spread-out targets that a prior can partly match. That is why scores on
all windows and on ≥ 10-vote windows are not comparable, and why every score must say which set it was computed on.
**From Phase 2 on we report CV on ≥ 10-vote windows** (the test set is believed to resemble high-vote data),
and keep the all-windows number for reference.

## 7. Save the numbers

In [ ]:
counts_out = pd.concat([counts, pd.Series({
    f"windows >= {HIGH_VOTES} votes": int(hi.sum()),
    f"windows < {HIGH_VOTES} votes": int((~hi).sum()),
    f"patients with any >= {HIGH_VOTES}-vote window": int(train.loc[hi, "patient_id"].nunique()),
})])
counts_out.to_csv(OUT_DIR / "dataset_counts.csv", header=["value"])
consensus_share.to_csv(OUT_DIR / "class_share_by_consensus.csv")
mean_soft.to_csv(OUT_DIR / "class_prior_mean_soft_label.csv")
agree_tab.to_csv(OUT_DIR / "agreement_categories.csv")
pd.concat([baselines, eeg_rows]).to_csv(OUT_DIR / "baselines_kl.csv")
print("Saved:", sorted(p.name for p in OUT_DIR.iterdir()))

## 8. Summary for the score log and the writeup

Fill in after running (copy from the tables above):

| Item | Value |
|---|---|
| Windows / EEGs / patients | 106800/ 17089/ 1950|
| Windows with ≥ 10 votes (share) | 39946 (37.4%) |
| KL uniform — all / ≥ 10 votes | 1.4023/1.0887 |
| KL class prior (CV by patient) — all / ≥ 10 votes | 1.2602/ 1.2102|
| Most common edge-case class pair | LRDA/Other : 2176|

**Score-log row:** `Ex1 | class prior, GroupKFold(5) by patient | CV KL all = … | CV KL ≥10 = … | no submission`

**Writeup hooks**
- *Background:* labels are vote distributions from a variable number of experts (two labelling rounds, visible in the
  vote histogram); experts agree only moderately (Jing et al. 2023), shown by the idealized/proto/edge split.
- *Results:* the uniform and class-prior rows are the first lines of the results table — the floor every model must beat.

**Answer for yourself before Exercise 2:** which classes are most often edge cases, and does that match the L/G and PD/RDA logic?